In [2]:
import pickle
import pandas as pd
import numpy as np
from collections import Counter, defaultdict
import matplotlib.pyplot as plt
import plotly.express as px

In [2]:
import pandas as pd
df = pd.read_csv("../Datasets/litnet.csv", index_col=0)
df

,in_tos_field,in_pkt,in_bytes,in_bytes_per_flow,out_bytes_per_flow,exporting_proc_id,src_add_141.98,src_add_158.129,src_add_193.219,src_add_83.171,...,src_port_443,src_port_771,src_port_Other,dest_port_53,dest_port_80,dest_port_123,dest_port_443,dest_port_771,dest_port_Other,attack_type
time_delta,,,,,,,,,,,,,,,,,,,,,
0.256,40,2,130,103,3,2,0,0,0,0,...,0,0,1,1,0,0,0,0,0,0
0.000,0,1,46,3,50,2,0,0,1,0,...,0,0,1,0,0,0,0,0,1,0
0.000,0,1,85,3,103,2,0,0,1,0,...,0,0,0,0,0,0,0,0,1,0
1.280,0,3,1115,170,3,2,0,0,0,0,...,0,0,1,0,0,0,1,0,0,0
1.536,0,16,1943,213,190,2,0,0,1,0,...,0,0,1,0,0,0,1,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
0.000,104,1,72,103,3,2,0,0,0,0,...,0,0,1,1,0,0,0,0,0,0
0.000,0,1,46,170,50,2,0,0,0,0,...,0,0,1,0,0,0,0,0,1,0
0.640,0,4,190,60,50,2,0,1,0,0,...,0,0,1,0,0,0,0,0,1,0


In [3]:
df.to_csv("../Datasets/litnet.csv", index=False)

In [3]:
import pandas as pd 
df = pd.read_csv("../TablaResultadosEnteraSinRuido.csv", index_col=0)
df["ratio"] = df["ratio"].fillna(0)
# df = df[df["n_classes"] != 2]

In [4]:
df

,Model,Accuracy,Precision,Recall,F1,dataset,ratio,sampler,seed,n_classes,nrows,nfeat
0,Random Forest,0.989565,0.494783,0.500000,0.497378,abalone-20_vs_8-9-10,0.0,original,0,2,1916,10
1,XGBoost,0.991304,0.995645,0.583333,0.640670,abalone-20_vs_8-9-10,0.0,original,0,2,1916,10
2,LightGBM,0.993043,0.872373,0.749121,0.798246,abalone-20_vs_8-9-10,0.0,original,0,2,1916,10
3,Random Forest,0.984348,0.691418,0.909637,0.759179,abalone-20_vs_8-9-10,0.6,smote,0,2,1916,10
4,XGBoost,0.977391,0.646163,0.906122,0.711624,abalone-20_vs_8-9-10,0.6,smote,0,2,1916,10
...,...,...,...,...,...,...,...,...,...,...,...,...
17458,XGBoost,1.000000,1.000000,1.000000,1.000000,wine_binary,0.8,borderline,29,2,1143,13
17459,LightGBM,1.000000,1.000000,1.000000,1.000000,wine_binary,0.8,borderline,29,2,1143,13
17460,Random Forest,1.000000,1.000000,1.000000,1.000000,wine_binary,1.0,borderline,29,2,1143,13
17461,XGBoost,1.000000,1.000000,1.000000,1.000000,wine_binary,1.0,borderline,29,2,1143,13


In [4]:
(df["F1"] == 1).sum()

np.int64(2039)

In [9]:
import pandas as pd
import plotly.express as px

# 1. PASO CRUCIAL: Seleccionar el mejor ratio para cada sampler en cada dataset/modelo
# Primero promediamos las 30 semillas para cada combinación
df_seeds_mean = df.groupby(['dataset', 'Model', 'sampler', 'ratio'])['F1'].mean().reset_index()

# Ahora buscamos cuál es el ratio que dio el F1 más alto para cada Sampler/Dataset/Modelo
idx_best_ratio = df_seeds_mean.groupby(['dataset', 'Model', 'sampler'])['F1'].idxmax()
df_best_configs = df_seeds_mean.loc[idx_best_ratio].reset_index(drop=True)

# 2. PROMEDIO ENTRE MODELOS (RF, XGB, LGBM)
# Para tener un único valor por Sampler y Dataset
df_agg = df_best_configs.groupby(['dataset', 'sampler'])['F1'].mean().reset_index()

# 3. CÁLCULO DEL RANKING POR DATASET
# (El rango 1 es para el F1 más alto)
df_agg['rank'] = df_agg.groupby('dataset')['F1'].rank(method='min', ascending=False)

# 4. AGREGACIÓN PARA EL GRÁFICO (Rango medio global)
df_plot = df_agg.groupby('sampler')['rank'].agg(['mean', 'sem']).reset_index()
df_plot = df_plot.sort_values('mean')

# --- VISUALIZACIÓN ---
fig = px.bar(
    df_plot,
    x='sampler',
    y='mean',
    error_y='sem',
    color='sampler',
    title="Ranking Medio de Samplers (Mejor Ratio seleccionado)",
    labels={'mean': 'Rango Medio (Menor es mejor)', 'sampler': 'Técnica de Muestreo'},
    color_discrete_sequence=px.colors.qualitative.Safe
)

fig.update_layout(
    yaxis=dict(autorange="reversed", title="Rango Medio (1º es el top)"),
    width=1000, height=600,
    showlegend=False
)

fig.show()

# --- PRINT DE RESULTADOS DETALLADOS ---
print("\n" + "="*50)
print("RANKING BASADO EN EL MEJOR RATIO PROMEDIO (30 SEMILLAS)")
print("="*50)

df_ranking_detail = df_agg.sort_values(['dataset', 'rank'])

for dataset in df_ranking_detail['dataset'].unique():
    print(f"\n📊 Dataset: {dataset}")
    subset = df_ranking_detail[df_ranking_detail['dataset'] == dataset]
    
    # Recuperamos qué ratio fue el ganador para informar en el print
    for _, row in subset.iterrows():
        # Buscamos el ratio que usamos (promediando entre los 3 modelos para ese sampler)
        ratio_info = df_best_configs[
            (df_best_configs['dataset'] == dataset) & 
            (df_best_configs['sampler'] == row['sampler'])
        ]['ratio'].unique()
        
        ratios_str = "/".join(map(str, ratio_info))
        print(f"  {int(row['rank'])}º - {row['sampler']:<15} | F1 Medio: {row['F1']:.4f} (Ratios usados: {ratios_str})")


RANKING BASADO EN EL MEJOR RATIO PROMEDIO (30 SEMILLAS)

📊 Dataset: abalone-20_vs_8-9-10
  1º - borderline      | F1 Medio: 0.6536 (Ratios usados: 1.0/0.6)
  2º - corsmote        | F1 Medio: 0.6511 (Ratios usados: 1.0/0.6/0.8)
  3º - smote           | F1 Medio: 0.6503 (Ratios usados: 0.8/0.6)
  4º - adasyn          | F1 Medio: 0.6437 (Ratios usados: 0.8/1.0)
  5º - original        | F1 Medio: 0.5846 (Ratios usados: 0.0)

📊 Dataset: cicids2017
  1º - corsmote        | F1 Medio: 0.9931 (Ratios usados: 1.0)
  2º - smote           | F1 Medio: 0.9929 (Ratios usados: 0.6/0.8)
  3º - original        | F1 Medio: 0.9929 (Ratios usados: 0.0)
  4º - borderline      | F1 Medio: 0.9926 (Ratios usados: 0.8/0.6)
  5º - adasyn          | F1 Medio: 0.9909 (Ratios usados: 0.6)

📊 Dataset: cleveland_0_vs_4
  1º - adasyn          | F1 Medio: 0.8053 (Ratios usados: 0.8/1.0)
  2º - smote           | F1 Medio: 0.8009 (Ratios usados: 1.0/0.8)
  3º - borderline      | F1 Medio: 0.7930 (Ratios usados: 0.6/1.0/

In [6]:
df_agg = df.groupby(['dataset', 'sampler'])['F1'].mean().reset_index()
df_agg['rank'] = df_agg.groupby('dataset')['F1'].rank(method='min', ascending=False)

df_plot = df_agg.groupby('sampler')['rank'].agg(['mean', 'sem']).reset_index()

df_plot = df_plot.sort_values('mean')

fig = px.bar(
    df_plot,
    x='sampler',
    y='mean',
    error_y='sem',               
    color='sampler',                
    color_discrete_sequence=px.colors.qualitative.Plotly,
    labels={'mean': 'Rango medio', 'sampler': 'Sampler'}
)

fig.update_layout(
    yaxis=dict(autorange="reversed"),
    width=1000, height=600,           
    coloraxis_showscale=False         
)

fig.update_traces(error_y=dict(width=10, thickness=1.5))

fig.show()
fig.write_image("../Comparativa_RangoMedio_Sampler.pdf", format="pdf")

print("\n" + "="*40)
print("RANKING DETALLADO POR DATASET")
print("="*40)

# Ordenamos para que la lectura sea lógica: primero por dataset y luego por posición en el ranking
df_ranking_detail = df_agg.sort_values(['dataset', 'rank'])

for dataset in df_ranking_detail['dataset'].unique():
    print(f"\n📊 Dataset: {dataset}")
    subset = df_ranking_detail[df_ranking_detail['dataset'] == dataset]
    
    for _, row in subset.iterrows():
        # Usamos .format o f-strings para alinear un poco el texto
        print(f"  {int(row['rank'])}º - {row['sampler']} (F1: {row['F1']:.4f})")


RANKING DETALLADO POR DATASET

📊 Dataset: abalone-20_vs_8-9-10
  1º - borderline (F1: 0.6492)
  2º - corsmote (F1: 0.6492)
  3º - smote (F1: 0.6460)
  4º - adasyn (F1: 0.6398)
  5º - original (F1: 0.5846)

📊 Dataset: cicids2017
  1º - corsmote (F1: 0.9930)
  2º - original (F1: 0.9929)
  3º - smote (F1: 0.9928)
  4º - borderline (F1: 0.9925)
  5º - adasyn (F1: 0.9905)

📊 Dataset: cleveland_0_vs_4
  1º - adasyn (F1: 0.8004)
  2º - smote (F1: 0.7975)
  3º - borderline (F1: 0.7857)
  4º - corsmote (F1: 0.7633)
  5º - original (F1: 0.7068)

📊 Dataset: credit
  1º - corsmote (F1: 0.8951)
  2º - borderline (F1: 0.8909)
  3º - smote (F1: 0.8864)
  4º - adasyn (F1: 0.8755)
  5º - original (F1: 0.8718)

📊 Dataset: diabetes
  1º - original (F1: 0.7251)
  2º - corsmote (F1: 0.7214)
  3º - smote (F1: 0.7203)
  4º - borderline (F1: 0.7188)
  5º - adasyn (F1: 0.7186)

📊 Dataset: flare-F
  1º - corsmote (F1: 0.6074)
  2º - smote (F1: 0.5948)
  3º - adasyn (F1: 0.5933)
  4º - borderline (F1: 0.5878)
 

In [6]:
fig = px.box(df, x="ratio", y="F1", color="sampler")
fig.show()

In [9]:
fig = px.box(df, y="F1", color="sampler", x="dataset")
fig.show()
fig.write_image("../ComparativaF1_por_dataset.pdf", format="pdf", width=1920)